# Phonological Density and Dependency Distance Minimization

This notebook demonstrates a computational linguistics experiment investigating the relationship between phonological density and dependency distance minimization across Universal Dependencies treebanks.

**Key components:**
- Computes word-space and phoneme-space dependency distances
- Correlates with phonological density indices
- Tests whether phonologically denser languages show stronger dependency distance minimization
- Includes baseline comparisons by word order type (SVO, SOV, VSO)

This is a demo notebook. The full analysis processes 55 languages across 10 language families.

In [ ]:
# Install dependencies following aii-colab pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install everywhere)
# epitran is used for phoneme conversion in the full script
_pip('epitran==0.11.0')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
# Imports
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

In [ ]:
# Data loading helper with GitHub URL + local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-45370e-phonotactic-constraint-on-dependency/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded demo data: {data['metadata']['n_successful']} languages, {data['metadata']['n_families']} families")
print(f"Datasets: {[d['dataset'] for d in data['datasets']]}")

## Configuration

All tunable parameters are defined here. These are set to the absolute minimum values that still produce meaningful output.

In [ ]:
# ============================================================
# CONFIG - All tunable parameters
# ============================================================

# Number of languages to process from the demo data
# Original script: 55 languages. Demo: 3 representative languages.
N_LANGS = 3

# Maximum sentences per language for processing
# Original script: all available. Demo: small subset for speed.
MAX_SENTENCES_PER_LANG = 10

# Statistical analysis parameters
N_BOOTSTRAP = 100          # Bootstrap iterations for confidence intervals
N_PERMUTATIONS = 100       # Permutation test iterations
RANDOM_SEED = 42           # For reproducibility

# Visualization settings
FIG_DPI = 150              # Resolution for saved figures
FIG_SIZE_SINGLE = (6, 4.5) # Single plot size
FIG_SIZE_MULTI = (10, 4)   # Multi-panel plot size

## Data Preparation

Extract the pre-computed results from the demo data and prepare them for analysis.

In [ ]:
# Extract successful results from demo data
successful_results = []
for dataset_info in data["datasets"][:N_LANGS]:
    for example in dataset_info["examples"]:
        output = json.loads(example["output"])
        result = {
            "dataset": dataset_info["dataset"],
            "input": example["input"],
            "output": output,
            "predict_baseline": json.loads(example["predict_baseline"]),
            "predict_our_method": json.loads(example["predict_our_method"]),
        }
        # Parse input for metadata
        # Format: "Language: xx, Family: YY, Word Order: Z, Phoneme Count: N, Phon Density: D"
        parts = example["input"].split(", ")
        for part in parts:
            if part.startswith("Language: "):
                result["lang_code"] = part.split(": ")[1]
            elif part.startswith("Family: "):
                result["language_family"] = part.split(": ")[1]
            elif part.startswith("Word Order: "):
                result["word_order"] = part.split(": ")[1]
            elif part.startswith("Phoneme Count: "):
                result["phoneme_count"] = int(part.split(": ")[1])
            elif part.startswith("Phon Density: "):
                result["phon_density"] = float(part.split(": ")[1])
        successful_results.append(result)

print(f"Prepared {len(successful_results)} results for analysis")
for r in successful_results:
    print(f"  {r['dataset']}: MDD={r['output']['mdd_word']['mean']:.3f}, phon_density={r['phon_density']:.3f}")

## Statistical Analysis

Run regression analysis and baseline comparisons on the prepared results.

In [ ]:
# ============================================================
# ANALYSIS FUNCTIONS
# ============================================================

def run_simple_regression(results):
    """Fallback simple regression without statsmodels."""
    valid = [r for r in results if "mdd_word" in r.get("output", {}) and r.get("output", {}).get("mdd_word", {}).get("mean", 0) > 0]
    if len(valid) < 3:
        return {"error": "too few valid results", "n": len(valid)}

    mdd_values = np.array([r["output"]["mdd_word"]["mean"] for r in valid])
    phon_density = np.array([r.get("phon_density", 0) for r in valid])

    # Simple OLS via numpy
    n = len(mdd_values)
    x_mean = np.mean(phon_density)
    y_mean = np.mean(mdd_values)

    ss_xy = np.sum((phon_density - x_mean) * (mdd_values - y_mean))
    ss_xx = np.sum((phon_density - x_mean) ** 2)

    if ss_xx == 0:
        return {"error": "zero variance in phon_density"}

    beta1 = ss_xy / ss_xx
    beta0 = y_mean - beta1 * x_mean

    # R-squared
    y_pred = beta0 + beta1 * phon_density
    ss_res = np.sum((mdd_values - y_pred) ** 2)
    ss_tot = np.sum((mdd_values - y_mean) ** 2)
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    regression_out = {
        "model": "SimpleOLS",
        "n_languages": n,
        "r_squared": float(r_squared),
        "phon_density_coefficient": float(beta1),
        "intercept": float(beta0),
    }

    # Spearman correlation
    try:
        spearman_r, spearman_p = stats.spearmanr(phon_density, mdd_values)
        regression_out["spearman_r"] = float(spearman_r)
        regression_out["spearman_p"] = float(spearman_p)
    except Exception:
        pass

    return regression_out


def run_baseline_comparison(results):
    """Baseline: compare word-order groups (SVO vs SOV vs VSO)."""
    valid = [r for r in results if "mdd_word" in r.get("output", {}) and r.get("output", {}).get("mdd_word", {}).get("mean", 0) > 0]

    groups = defaultdict(list)
    for r in valid:
        wo = r.get("word_order", "Other")
        groups[wo].append(r["output"]["mdd_word"]["mean"])

    comparison = {}
    for wo, values in groups.items():
        comparison[wo] = {
            "n": len(values),
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
            "median": float(np.median(values)),
        }

    # ANOVA if enough groups
    if len(groups) >= 2:
        group_lists = [np.array(v) for v in groups.values()]
        try:
            f_stat, p_val = stats.f_oneway(*group_lists)
            comparison["anova_f"] = float(f_stat)
            comparison["anova_p"] = float(p_val)
        except Exception:
            pass

    # Kruskal-Wallis non-parametric
    if len(groups) >= 2:
        try:
            h_stat, h_p = stats.kruskal(*[np.array(v) for v in groups.values()])
            comparison["kruskal_h"] = float(h_stat)
            comparison["kruskal_p"] = float(h_p)
        except Exception:
            pass

    return comparison


# Run analyses
print("Running regression analysis...")
regression_results = run_simple_regression(successful_results)
print(f"R²: {regression_results.get('r_squared', 'N/A'):.4f}")
print(f"Spearman r: {regression_results.get('spearman_r', 'N/A'):.4f}")
print(f"Phon density coefficient: {regression_results.get('phon_density_coefficient', 'N/A'):.4f}")

print("\nRunning baseline comparison...")
baseline_results = run_baseline_comparison(successful_results)
print("Baseline comparison by word order:")
for wo, stats_ in baseline_results.items():
    if wo not in ["anova_f", "anova_p", "kruskal_h", "kruskal_p"]:
        print(f"  {wo}: n={stats_['n']}, mean MDD={stats_['mean']:.3f}")

## Visualization

Generate publication-quality figures from the analysis results.

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================

def generate_figures(results, regression_results, baseline_results, out_dir):
    """Generate publication-quality figures."""
    out_dir.mkdir(exist_ok=True)
    fig_files = []

    valid = [r for r in results if "mdd_word" in r.get("output", {}) and r.get("output", {}).get("mdd_word", {}).get("mean", 0) > 0]
    if len(valid) < 3:
        print("Too few results for figures")
        return fig_files

    # Extract data
    phon_density = [r.get("phon_density", 0) for r in valid]
    mdd_word = [r["output"]["mdd_word"]["mean"] for r in valid]
    mdd_phoneme = [r.get("output", {}).get("mdd_phoneme", {}).get("mean", 0) or 0 for r in valid]
    families = [r.get("language_family", "Unknown") for r in valid]
    word_orders = [r.get("word_order", "SVO") for r in valid]

    # Color by word order
    wo_colors = {"SVO": "#4C72B0", "SOV": "#DD8452", "VSO": "#55A868"}
    colors = [wo_colors.get(wo, "#888888") for wo in word_orders]

    # Figure 1: Scatter plot - Phonological Density vs MDD
    try:
        fig, ax = plt.subplots(figsize=FIG_SIZE_SINGLE)
        ax.scatter(phon_density, mdd_word, c=colors, s=60, alpha=0.7, edgecolors='black', linewidth=0.5)

        # Regression line
        if len(phon_density) > 2:
            z = np.polyfit(phon_density, mdd_word, 1)
            p = np.poly1d(z)
            x_line = np.linspace(min(phon_density), max(phon_density), 100)
            ax.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=1.5,
                    label=f"OLS: β={z[0]:.3f}")

        # Color legend
        from matplotlib.lines import Line2D
        legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor=wo_colors[wo],
                                  markersize=8, label=wo) for wo in sorted(set(word_orders))]
        ax.legend(handles=legend_elements, title="Word Order", loc="lower right", framealpha=0.9)
        ax.set_xlabel("Phonological Density (normalized phoneme count)")
        ax.set_ylabel("Mean Dependency Distance (word-space)")
        ax.set_title(f"Phonological Density vs Dependency Distance\n(n={len(valid)} languages)")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig_path = out_dir / "fig1_density_vs_mdd.pdf"
        fig.savefig(fig_path, dpi=FIG_DPI)
        plt.close(fig)
        fig_files.append(str(fig_path))
        print(f"Saved {fig_path}")
    except Exception as e:
        print(f"Figure 1 failed: {e}")

    # Figure 2: Word-space vs Phoneme-space MDD
    try:
        phon_valid = [(pw, pm) for pw, pm in zip(mdd_word, mdd_phoneme) if pm > 0]
        if len(phon_valid) > 3:
            pw_list, pm_list = zip(*phon_valid)
            fig, ax = plt.subplots(figsize=FIG_SIZE_SINGLE)
            ax.scatter(pw_list, pm_list, c=colors[:len(pw_list)], s=50, alpha=0.7,
                       edgecolors='black', linewidth=0.5)
            # Diagonal reference
            max_val = max(max(pw_list), max(pm_list))
            ax.plot([0, max_val], [0, max_val], "k--", alpha=0.5, linewidth=1)
            ax.set_xlabel("Word-space MDD")
            ax.set_ylabel("Phoneme-space MDD")
            ax.set_title(f"Word-space vs Phoneme-space Dependency Distance\n(n={len(pw_list)})")
            ax.grid(True, alpha=0.3)
            fig.tight_layout()
            fig_path = out_dir / "fig2_word_vs_phoneme_mdd.pdf"
            fig.savefig(fig_path, dpi=FIG_DPI)
            plt.close(fig)
            fig_files.append(str(fig_path))
            print(f"Saved {fig_path}")
    except Exception as e:
        print(f"Figure 2 failed: {e}")

    # Figure 3: Functional vs Lexical MDD
    try:
        fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE_MULTI)
        # Functional
        func_mdd = [r.get("output", {}).get("mdd_word_functional", {}).get("mean", 0) for r in valid]
        valid_mask = [m > 0 for m in func_mdd]
        axes[0].scatter([phon_density[i] for i, v in enumerate(valid_mask) if v],
                        [func_mdd[i] for i, v in enumerate(valid_mask) if v],
                        c=[colors[i] for i, v in enumerate(valid_mask) if v],
                        s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
        axes[0].set_xlabel("Phonological Density")
        axes[0].set_ylabel("Functional MDD")
        axes[0].set_title("Functional Dependencies")
        axes[0].grid(True, alpha=0.3)
        # Lexical
        lex_mdd = [r.get("output", {}).get("mdd_word_lexical", {}).get("mean", 0) for r in valid]
        valid_mask2 = [m > 0 for m in lex_mdd]
        axes[1].scatter([phon_density[i] for i, v in enumerate(valid_mask2) if v],
                        [lex_mdd[i] for i, v in enumerate(valid_mask2) if v],
                        c=[colors[i] for i, v in enumerate(valid_mask2) if v],
                        s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
        axes[1].set_xlabel("Phonological Density")
        axes[1].set_ylabel("Lexical MDD")
        axes[1].set_title("Lexical Dependencies")
        axes[1].grid(True, alpha=0.3)
        fig.suptitle("Functional vs Lexical MDD by Phonological Density")
        fig.tight_layout()
        fig_path = out_dir / "fig3_functional_vs_lexical.pdf"
        fig.savefig(fig_path, dpi=FIG_DPI)
        plt.close(fig)
        fig_files.append(str(fig_path))
        print(f"Saved {fig_path}")
    except Exception as e:
        print(f"Figure 3 failed: {e}")

    return fig_files


# Generate figures
print("\nGenerating figures...")
fig_dir = Path("figures")
fig_files = generate_figures(successful_results, regression_results, baseline_results, fig_dir)
print(f"Generated {len(fig_files)} figures")

## Summary

Print the key results from the analysis.

In [ ]:
# Print summary
print("=" * 60)
print("RESULTS SUMMARY")
print(f"Languages processed: {len(successful_results)}")
print(f"Families: {len(set(r.get('language_family', 'Unknown') for r in successful_results))}")
print(f"Regression R²: {regression_results.get('r_squared', 'N/A'):.4f}")
print(f"Phon density coefficient: {regression_results.get('phon_density_coefficient', 'N/A'):.4f}")
print(f"Spearman r: {regression_results.get('spearman_r', 'N/A'):.4f}")
print(f"Spearman p: {regression_results.get('spearman_p', 'N/A'):.4f}")
print(f"Figures generated: {len(fig_files)}")
print("=" * 60)

# Display regression results table
print("\nRegression Results:")
print(f"  Model: {regression_results.get('model', 'N/A')}")
print(f"  R-squared: {regression_results.get('r_squared', 0):.4f}")
print(f"  Phon density coefficient: {regression_results.get('phon_density_coefficient', 0):.4f}")
print(f"  Intercept: {regression_results.get('intercept', 0):.4f}")
if 'spearman_r' in regression_results:
    print(f"  Spearman r: {regression_results['spearman_r']:.4f} (p={regression_results.get('spearman_p', 1):.4f})")

print("\nBaseline Comparison (Word Order):")
for wo, stats_ in baseline_results.items():
    if wo not in ["anova_f", "anova_p", "kruskal_h", "kruskal_p"]:
        print(f"  {wo}: n={stats_['n']}, mean MDD={stats_['mean']:.3f}, std={stats_['std']:.3f}")
if 'anova_f' in baseline_results:
    print(f"  ANOVA: F={baseline_results['anova_f']:.4f}, p={baseline_results['anova_p']:.4f}")
if 'kruskal_h' in baseline_results:
    print(f"  Kruskal-Wallis: H={baseline_results['kruskal_h']:.4f}, p={baseline_results['kruskal_p']:.4f}")